# Transfer Learning & Fine-Tuning — AI Lab Instructor Notebook

*Deep Learning · Hard*



7 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Transfer Learning & Fine-Tuning (NumPy) — Student Lab

We simulate transfer learning with a small MLP on `sklearn.datasets.load_digits`.

Source task (pretrain): **even vs odd**.
Target task (fine-tune): **10-class digit classification**.

## Section 0 — Data splits (simulate low-data target)

We create:
- a large pool for source pretraining
- a *small* target training set (e.g., 200 samples)
- a held-out validation set

FAANG gotcha: standardize using **train stats only** to avoid leakage.

In [ ]:
$$10

In [ ]:
# Checks
check('shapes', X_source_z.shape[1] == 64 and X_ttrain_z.shape[1] == 64 and X_val_z.shape[1] == 64)

**Why this works.** $f

## Task 2: Train LogisticRegression baseline
*Section: Baseline: LogisticRegression on small target set*

## Section 1 — Baseline: LogisticRegression on small target set

This baseline is intentionally strong and hard to beat with tiny models.

### Task 1.1
Train `LogisticRegression` on the small target train split and report train/validation accuracy.

In [ ]:
lr_baseline = LogisticRegression(max_iter=5000)
lr_baseline.fit(X_ttrain_z, y10_ttrain)
print('LR baseline train acc', float(lr_baseline.score(X_ttrain_z, y10_ttrain)))
print('LR baseline val   acc', float(lr_baseline.score(X_val_z, y10_val)))

In [ ]:
# Checks
check('baseline_sane', np.isfinite(float(lr_baseline.score(X_val_z, y10_val))))

## Task 3: Define MLP forward/backward + train loop
*Section: A small MLP (feature extractor + head)*

## Section 2 — A small MLP (feature extractor + head)

We define MLP helper functions used in later sections.

### Task 2.1
Implement `init_mlp`, `forward`, `backward`, `accuracy`, and `train_mlp`.

In [ ]:
$$13

In [ ]:
# Checks
check('helpers_defined', all(k in globals() for k in ['init_mlp','train_mlp','accuracy']))

## Task 4: Train an MLP from scratch on `y10` using only the small target training set.
*Section: Train from scratch on the small target set*

## Section 3 — Train from scratch on the small target set

### Task 3.1
Train an MLP from scratch on `y10` using only the small target training set.

In [ ]:
scratch = init_mlp(64, 64, 10, seed=0)
scratch = train_mlp(scratch, X_ttrain_z, y10_ttrain, X_val_z, y10_val, lr=0.15, wd=1e-4, epochs=40, batch_size=64, freeze_feature=False, seed=0)
print('scratch final train acc', accuracy(scratch, X_ttrain_z, y10_ttrain))
print('scratch final val   acc', accuracy(scratch, X_val_z, y10_val))
check('scratch_runs', np.isfinite(accuracy(scratch, X_val_z, y10_val)))

In [ ]:
# Checks
check('scratch_runs', np.isfinite(accuracy(scratch, X_val_z, y10_val)))

## Task 5: Train a 2-class MLP on the source task.
*Section: Pretrain on the source task (even vs odd)*

## Section 4 — Pretrain on the source task (even vs odd)

We pretrain the same MLP but with a 2-class head.

### Task 4.1
Train a 2-class MLP on the source task.

In [ ]:
pre = init_mlp(64, 64, 2, seed=1)
pre = train_mlp(pre, X_source_z, yp_source, X_val_z, yp_val, lr=0.1, wd=1e-4, epochs=30, batch_size=128, freeze_feature=False, seed=1)
print('pretrain source train acc', accuracy(pre, X_source_z, yp_source))
print('pretrain source val   acc', accuracy(pre, X_val_z, yp_val))
check('pretrain_source_good', accuracy(pre, X_val_z, yp_val) > 0.8)

In [ ]:
# Checks
check('pretrain_source_good', accuracy(pre, X_val_z, yp_val) > 0.8)

**Why this works.** ## Baseline: LogisticRegression

Rationale: strong linear baseline; transfer should beat scratch MLP more reliably than it beats LR.

## Task 6: Frozen features: update only `(W2,b2)` (new head).
*Section: Transfer: frozen features vs fine-tuning*

## Section 5 — Transfer: frozen features vs fine-tuning

We reuse `(W1, b1)` from pretraining and create a new 10-class head.

### Task 5.1
Frozen features: update only `(W2,b2)` (new head).

### Task 5.2
Fine-tuning: update all parameters starting from the pretrained init (usually smaller LR).

## Section 6 — Compare results
Which is best on validation: scratch, frozen, or finetune?

In [ ]:
$$15

In [ ]:
# Checks
check('transfer_runs', np.isfinite(accuracy(finetune, X_val_z, y10_val)))
check('acc_sane', 0.5 <= results[max(results, key=results.get)] <= 1.0)

**Why this works.** ## Model + training utilities

Rationale: We separate feature extractor and head so we can freeze/unfreeze easily.

## Task 7: Compare transferred frozen features against random frozen features.
*Section: Control experiment: pretrained frozen vs random frozen*

## Section 7 — Control experiment: pretrained frozen vs random frozen

This task isolates the value of transfer.

A frozen-head model can perform well for two different reasons:
- the reused features are genuinely useful
- or we simply reduced the number of trainable parameters

To separate those effects, compare:
- **pretrained frozen**: reuse `(W1, b1)` from parity pretraining, train only the new head
- **random frozen**: keep a random `(W1, b1)` fixed, train only the head

### Task 7.1
Run both settings across several seeds and compare the **mean** validation accuracy.

Expected interpretation: if pretrained frozen beats random frozen on average, the transferred features are carrying useful signal.

**Hints**
- Use the same optimizer settings for both frozen models so the comparison is fair.
- One seed can be noisy. Compare the mean over multiple seeds.

In [ ]:
# No source solution — left as an exercise.